# AdaBoost 算法迭代过程示例

本文档是 [AdaBoost 计算示例.md](./Adaboost%20计算示例.md) 的配套代码，用于复现文档中的手算推导过程。

## 1. 初始化数据与环境

In [1]:
import numpy as np

# 题目给定数据集（x 为一维特征，y 为 {-1, +1} 标签）
x = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
y = np.array([1, 1, 1, -1, -1, -1, 1, 1, 1, -1], dtype=float)
n = y.size

print("样本特征 x:", x)
print("样本标签 y:", y)

样本特征 x: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]
样本标签 y: [ 1.  1.  1. -1. -1. -1.  1.  1.  1. -1.]


## 2. 定义候选弱分类器

根据题目设定，我们有三个候选弱分类器：
- $G_1(x)$: $x < 2.5$ 为 $+1$，否则为 $-1$
- $G_2(x)$: $x < 8.5$ 为 $+1$，否则为 $-1$
- $G_3(x)$: $x \ge 5.5$ 为 $+1$，否则为 $-1$

In [2]:
# 候选弱分类器（向量化实现，返回形状为 (n,) 的预测向量）
def G1(x_vec: np.ndarray) -> np.ndarray:
    # 若 x < 2.5 则输出 +1，否则输出 -1
    return np.where(x_vec < 2.5, 1.0, -1.0)

def G2(x_vec: np.ndarray) -> np.ndarray:
    # 若 x < 8.5 则输出 +1，否则输出 -1
    return np.where(x_vec < 8.5, 1.0, -1.0)

def G3(x_vec: np.ndarray) -> np.ndarray:
    # 若 x >= 5.5 则输出 +1，否则输出 -1
    return np.where(x_vec >= 5.5, 1.0, -1.0)

candidates = [("G1", G1), ("G2", G2), ("G3", G3)]

## 3. AdaBoost 迭代训练过程

执行 3 轮迭代，每轮选择加权误差最小的弱分类器，并更新样本权重。

In [3]:
# 初始化样本权重分布 D1（均匀分布）
D = np.full(shape=n, fill_value=1.0 / n, dtype=float)

chosen = []
alphas = []

print("初始权重 D1:", D)
print("-" * 60)

for t in range(1, 4):
    # 1) 在候选集合中选出加权误差最小的弱分类器（并列取列表中更靠前者）
    best_name = None
    best_pred = None
    best_error = None
    best_idx = None

    for idx, (name, G) in enumerate(candidates):
        pred = G(x)
        # 计算加权误差: sum(D_i * I(y_pred != y_true))
        error = float(np.sum(D[pred != y]))
        
        # 寻找最小误差（并列时保持原有顺序，即取索引较小者）
        if best_error is None or error < best_error - 1e-12 or (abs(error - best_error) <= 1e-12 and idx < best_idx):
            best_name = name
            best_pred = pred
            best_error = error
            best_idx = idx

    # 2) 计算 alpha（为数值稳定性，避免误差为 0 或 1）
    eps = 1e-12
    clipped_error = min(max(best_error, eps), 1.0 - eps)
    alpha = 0.5 * np.log((1.0 - clipped_error) / clipped_error)

    # 3) 更新权重并归一化：D_{t+1}(i) ∝ D_t(i) * exp(-alpha * y_i * pred_i)
    D = D * np.exp(-alpha * y * best_pred)
    D = D / np.sum(D)

    chosen.append(best_name)
    alphas.append(alpha)

    # 4) 输出便于核对的中间结果
    print(f"第 {t} 轮选择: {best_name}, 加权误差: {best_error:.4f}, alpha: {alpha:.4f}")
    print("更新后的 D:", np.round(D, 4))
    print("-" * 60)

初始权重 D1: [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
------------------------------------------------------------
第 1 轮选择: G1, 加权误差: 0.3000, alpha: 0.4236
更新后的 D: [0.0714 0.0714 0.0714 0.0714 0.0714 0.0714 0.1667 0.1667 0.1667 0.0714]
------------------------------------------------------------
第 2 轮选择: G2, 加权误差: 0.2143, alpha: 0.6496
更新后的 D: [0.0455 0.0455 0.0455 0.1667 0.1667 0.1667 0.1061 0.1061 0.1061 0.0455]
------------------------------------------------------------
第 3 轮选择: G3, 加权误差: 0.1818, alpha: 0.7520
更新后的 D: [0.125  0.125  0.125  0.1019 0.1019 0.1019 0.0648 0.0648 0.0648 0.125 ]
------------------------------------------------------------


## 4. 验证最终强分类器效果

In [4]:
# 最终强分类器在训练集上的预测（sign(Σ alpha_t G_t(x))）
score = np.zeros_like(y)
for name, alpha in zip(chosen, alphas):
    G = dict(candidates)[name]
    score += alpha * G(x)

y_pred = np.where(score >= 0, 1.0, -1.0)
train_error = float(np.mean(y_pred != y))

print("最终选择顺序:", chosen)
print("最终 alpha:", [round(float(a), 4) for a in alphas])
print("最终预测结果:", y_pred)
print("真实标签:", y)
print("训练误差:", train_error)

最终选择顺序: ['G1', 'G2', 'G3']
最终 alpha: [0.4236, 0.6496, 0.752]
最终预测结果: [ 1.  1.  1. -1. -1. -1.  1.  1.  1. -1.]
真实标签: [ 1.  1.  1. -1. -1. -1.  1.  1.  1. -1.]
训练误差: 0.0
